# SafeBite analytics exploration

This notebook reads the Power BI CSV extracts produced from the SafeBite demo database. Run `npm run analytics:datasets` from the repository root before opening it.

**Synthetic demo data.** These figures are not official government statistics and do not imply government endorsement. Risk scores are deterministic weighted policy rules, not a trained machine-learning model and not a probability of illness.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# The notebook lives in analytics/python/notebooks.
PACKAGE_ROOT = Path.cwd().parent
REPO_ROOT = PACKAGE_ROOT.parents[1]
sys.path.insert(0, str(PACKAGE_ROOT))

from safebite_analytics.loader import load_datasets
from safebite_analytics.kpis import compute_kpis, kpi_frame, resolve_period
from safebite_analytics.risk import district_risk, factor_profile, latest_risk, risk_movers, tier_summary
from safebite_analytics.quality import compare_with_export, evaluate_quality

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

data = load_datasets(REPO_ROOT / "public" / "powerbi")
print(f"Extract generated: {data.generated_at}")
print(f"Businesses: {len(data.businesses):,} | inspections: {len(data.inspections):,} | complaints: {len(data.complaints):,} | snapshots: {len(data.snapshots):,}")

## 1. Governed KPIs (last 90 days)

The period uses Asia/Kolkata day boundaries converted to UTC evidence, matching the dashboard. The previous period is the immediately preceding window of equal length.

In [ ]:
period = resolve_period("last90", now=data.generated_at)
print(period.label, "vs", period.previous_label)

kpis = compute_kpis(data, period)
kpi_frame(kpis)[["name", "current_display", "previous_display", "percent_change", "assessment"]]

## 2. Risk register at the extract date

Tier C is high risk and tier D is critical. The score is capped at 100 and built from five weighted policy factors: violations 40, recent complaints 25, inspection timeliness 15, document compliance 10 and category baseline 10.

In [ ]:
tier_summary(data, data.generated_at)

In [ ]:
latest_risk(data, data.generated_at)[
    ["name", "district", "category", "score", "tier", "topFactor", "topFactorPoints"]
].head(10)

## 3. Factor contribution profile

Average utilisation shows which risk factor consumes most of its policy budget across rated premises. This is an explainability view, not a model feature-importance chart.

In [ ]:
factor_profile(data, data.generated_at)

In [ ]:
risk_movers(data, data.generated_at, days=30, limit=10)[
    ["name", "district", "priorScore", "currentScore", "scoreChange", "direction"]
]

## 4. District comparison

Small districts should be read with care: a single high-risk premises moves the share sharply.

In [ ]:
district_risk(data, data.generated_at)

## 5. Data quality reconciliation

The app's Prisma-level engine is authoritative. This view re-evaluates the rules that can be tested from the CSV extract and compares counts with the exported findings. Rules needing checklists, documents or suppliers are marked not evaluable.

In [ ]:
report = evaluate_quality(data)
print(f"Rules evaluated: {report.rules_evaluated} | issues: {report.issues}")
pd.DataFrame(report.findings)[["ruleId", "rule", "severity", "affectedRecords"]]

In [ ]:
comparison = compare_with_export(data, report)
comparison[comparison["status"] != "Not evaluable"]

## 6. Caveats

- Complaint SLA durations are calendar days, not business days.
- Historical risk values only use snapshots recorded on or before the date. They are never back-filled from today's register.
- The dataset has no acknowledgement, on-site inspection duration, laboratory, report-approval or enforcement timestamps.
- Complaint narratives, citizen identities, passwords and emails are excluded from the extracts.
- The demo is synthetic. Re-run `npm run analytics:smoke` for the authoritative current figures.